In [1]:
# ============================================================
# FER2013 - EfficientNet-B0 Advanced Fine-Tuning
# Fully Optimized Final Version
# ============================================================

import os
import time
import copy
import random
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.cuda.amp import autocast, GradScaler

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

from torchvision import transforms, models
from torchvision.datasets import ImageFolder

from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "EfficientNetB0_AdvancedFineTune"

TRAIN_DIR = "/kaggle/input/datasets/ananthu017/emotion-detection-fer/train"
TEST_DIR = "/kaggle/input/datasets/ananthu017/emotion-detection-fer/test"

OUTPUT_DIR = f"./{MODEL_NAME}_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 7

NUM_EPOCHS = 35

# Phase strategy
HEAD_ONLY_EPOCHS = 3
PARTIAL_UNFREEZE_EPOCHS = 8

# Learning rates
LR_HEAD = 1e-3
LR_PARTIAL = 1e-4
LR_FULL = 1e-5

WEIGHT_DECAY = 1e-4

EARLY_STOPPING_PATIENCE = 8

RANDOM_SEED = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


set_seed(RANDOM_SEED)

# ============================================================
# TRANSFORMS
# ============================================================

train_transform = transforms.Compose([

    transforms.Grayscale(num_output_channels=3),

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(15),

    transforms.RandomAffine(
        degrees=10,
        translate=(0.08, 0.08),
        scale=(0.9, 1.1)
    ),

    transforms.ColorJitter(
        brightness=0.25,
        contrast=0.25
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([

    transforms.Grayscale(num_output_channels=3),

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ============================================================
# DATASET
# ============================================================

class FERDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.dataset = ImageFolder(root=root_dir)

        self.transform = transform

    def __len__(self):

        return len(self.dataset)

    def __getitem__(self, idx):

        image, label = self.dataset[idx]

        if self.transform:
            image = self.transform(image)

        return image, label


# ============================================================
# LOAD DATASETS
# ============================================================

train_dataset = FERDataset(
    root_dir=TRAIN_DIR,
    transform=train_transform
)

test_dataset = FERDataset(
    root_dir=TEST_DIR,
    transform=test_transform
)

class_names = train_dataset.dataset.classes

# ============================================================
# TARGETS
# ============================================================

targets = [
    label
    for _, label in train_dataset.dataset.samples
]

# ============================================================
# CLASS WEIGHTS
# ============================================================

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(targets),
    y=targets
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(DEVICE)

# ============================================================
# WEIGHTED RANDOM SAMPLER
# ============================================================

class_sample_count = np.array([
    len(np.where(np.array(targets) == t)[0])
    for t in np.unique(targets)
])

weight = 1. / class_sample_count

samples_weight = np.array([
    weight[t]
    for t in targets
])

samples_weight = torch.from_numpy(
    samples_weight
).double()

sampler = WeightedRandomSampler(
    samples_weight,
    len(samples_weight)
)

# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ============================================================
# MODEL
# ============================================================

class EfficientNetB0FER(nn.Module):

    def __init__(self, num_classes=7):

        super(EfficientNetB0FER, self).__init__()

        # ----------------------------------------------------
        # IMPORTANT FIX:
        # weights=None prevents internet download issue
        # ----------------------------------------------------

        self.backbone = models.efficientnet_b0(
            weights=None
        )

        feature_dim = self.backbone.classifier[1].in_features

        self.backbone.classifier = nn.Sequential(

            nn.Dropout(0.5),

            nn.Linear(feature_dim, 512),

            nn.BatchNorm1d(512),

            nn.ReLU(inplace=True),

            nn.Dropout(0.4),

            nn.Linear(512, 256),

            nn.BatchNorm1d(256),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        return self.backbone(x)

# ============================================================
# FREEZE STRATEGIES
# ============================================================

def freeze_all_backbone(model):

    for param in model.backbone.features.parameters():

        param.requires_grad = False

    for param in model.backbone.classifier.parameters():

        param.requires_grad = True


def partial_unfreeze(model):

    for block in model.backbone.features[:4]:

        for param in block.parameters():

            param.requires_grad = False

    for block in model.backbone.features[4:]:

        for param in block.parameters():

            param.requires_grad = True

    for param in model.backbone.classifier.parameters():

        param.requires_grad = True


def full_unfreeze(model):

    for param in model.backbone.parameters():

        param.requires_grad = True

# ============================================================
# METRICS
# ============================================================

def compute_metrics(y_true, y_pred):

    return {

        "accuracy": accuracy_score(y_true, y_pred),

        "precision": precision_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average='macro',
            zero_division=0
        ),

        "per_class_f1": f1_score(
            y_true,
            y_pred,
            average=None,
            zero_division=0
        )
    }

# ============================================================
# PARAMETER COUNT
# ============================================================

def count_parameters(model):

    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total_params, trainable_params

# ============================================================
# TRAIN FUNCTION
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler
):

    model.train()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, leave=False):

        images = images.to(DEVICE)

        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        with autocast(enabled=torch.cuda.is_available()):

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

    epoch_loss = (
        running_loss / len(loader.dataset)
    )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    return epoch_loss, accuracy

# ============================================================
# VALIDATION FUNCTION
# ============================================================

def evaluate(model, loader, criterion):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in tqdm(loader, leave=False):

            images = images.to(DEVICE)

            labels = labels.to(DEVICE)

            with autocast(enabled=torch.cuda.is_available()):

                outputs = model(images)

                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    epoch_loss = (
        running_loss / len(loader.dataset)
    )

    metrics = compute_metrics(
        all_labels,
        all_preds
    )

    return (
        epoch_loss,
        metrics,
        all_labels,
        all_preds
    )

# ============================================================
# INITIALIZE MODEL
# ============================================================

model = EfficientNetB0FER(
    num_classes=NUM_CLASSES
).to(DEVICE)

# ============================================================
# PHASE 1 - TRAIN HEAD ONLY
# ============================================================

freeze_all_backbone(model)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.1
)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS
)

scaler = GradScaler()

best_model_wts = copy.deepcopy(
    model.state_dict()
)

best_f1 = 0.0

early_stop_counter = 0

history = []

# ============================================================
# TRAINING LOOP
# ============================================================

print("\n================================================")
print("Starting Advanced Fine-Tuning")
print("================================================")

start_training_time = time.time()

for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")

    # ========================================================
    # PHASE 2 - PARTIAL UNFREEZE
    # ========================================================

    if epoch == HEAD_ONLY_EPOCHS:

        print("\nPartial unfreezing...\n")

        partial_unfreeze(model)

        optimizer = optim.AdamW(

            filter(
                lambda p: p.requires_grad,
                model.parameters()
            ),

            lr=LR_PARTIAL,

            weight_decay=WEIGHT_DECAY
        )

    # ========================================================
    # PHASE 3 - FULL UNFREEZE
    # ========================================================

    if epoch == PARTIAL_UNFREEZE_EPOCHS:

        print("\nFull unfreezing...\n")

        full_unfreeze(model)

        optimizer = optim.AdamW(

            model.parameters(),

            lr=LR_FULL,

            weight_decay=WEIGHT_DECAY
        )

    train_loss, train_acc = train_one_epoch(

        model,
        train_loader,
        criterion,
        optimizer,
        scaler
    )

    scheduler.step()

    test_loss, test_metrics, _, _ = evaluate(

        model,
        test_loader,
        criterion
    )

    history.append({

        "epoch": epoch + 1,

        "train_loss": train_loss,

        "test_loss": test_loss,

        "train_accuracy": train_acc,

        "test_accuracy": test_metrics["accuracy"],

        "weighted_f1": test_metrics["weighted_f1"],

        "macro_f1": test_metrics["macro_f1"]
    })

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Test Accuracy: {test_metrics['accuracy']:.4f} | "
        f"Weighted F1: {test_metrics['weighted_f1']:.4f}"
    )

    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if test_metrics["weighted_f1"] > best_f1:

        best_f1 = test_metrics["weighted_f1"]

        best_model_wts = copy.deepcopy(
            model.state_dict()
        )

        torch.save(

            model.state_dict(),

            os.path.join(
                OUTPUT_DIR,
                f"{MODEL_NAME}_best.pth"
            )
        )

        early_stop_counter = 0

    else:

        early_stop_counter += 1

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if early_stop_counter >= EARLY_STOPPING_PATIENCE:

        print("\nEarly stopping triggered.\n")

        break

# ============================================================
# LOAD BEST MODEL
# ============================================================

model.load_state_dict(best_model_wts)

# ============================================================
# FINAL EVALUATION
# ============================================================

print("\n================================================")
print("Final Evaluation")
print("================================================")

test_loss, test_metrics, y_true, y_pred = evaluate(

    model,
    test_loader,
    criterion
)

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

report = classification_report(

    y_true,
    y_pred,

    target_names=class_names,

    digits=4
)

print(report)

with open(

    os.path.join(
        OUTPUT_DIR,
        "classification_report.txt"
    ),

    "w"
) as f:

    f.write(report)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred
)

cm_normalized = (
    cm.astype("float")
    / cm.sum(axis=1)[:, np.newaxis]
)

plt.figure(figsize=(10, 8))

sns.heatmap(

    cm_normalized,

    annot=True,

    fmt=".2f",

    cmap="Blues",

    xticklabels=class_names,

    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("True")

plt.title(
    "Normalized Confusion Matrix"
)

plt.tight_layout()

plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "normalized_confusion_matrix.png"
    ),

    dpi=300
)

plt.close()

# ============================================================
# INDIVIDUAL BINARY CONFUSION MATRICES
# ============================================================

for i, class_name in enumerate(class_names):

    binary_true = [
        1 if label == i else 0
        for label in y_true
    ]

    binary_pred = [
        1 if pred == i else 0
        for pred in y_pred
    ]

    binary_cm = confusion_matrix(
        binary_true,
        binary_pred
    )

    plt.figure(figsize=(6, 5))

    sns.heatmap(

        binary_cm,

        annot=True,

        fmt='d',

        cmap='Blues',

        xticklabels=[
            f"Not {class_name}",
            class_name
        ],

        yticklabels=[
            f"Not {class_name}",
            class_name
        ]
    )

    plt.xlabel("Predicted")
    plt.ylabel("True")

    plt.title(
        f"{class_name.capitalize()} vs Rest"
    )

    plt.tight_layout()

    plt.savefig(

        os.path.join(
            OUTPUT_DIR,
            f"{class_name}_binary_confusion_matrix.png"
        ),

        dpi=300
    )

    plt.close()

# ============================================================
# PER-CLASS F1 SCORE
# ============================================================

per_class_f1 = test_metrics["per_class_f1"]

plt.figure(figsize=(10, 6))

bars = plt.bar(
    class_names,
    per_class_f1
)

plt.ylim(0, 1)

plt.ylabel("F1 Score")

plt.title(
    "Per-Class F1 Score"
)

for bar, score in zip(bars, per_class_f1):

    plt.text(

        bar.get_x() + bar.get_width()/2,

        score + 0.01,

        f"{score:.3f}",

        ha='center'
    )

plt.tight_layout()

plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "per_class_f1_score.png"
    ),

    dpi=300
)

plt.close()

# ============================================================
# PER-CLASS ACCURACY
# ============================================================

per_class_accuracy = (
    cm.diagonal()
    / cm.sum(axis=1)
)

plt.figure(figsize=(10, 6))

bars = plt.bar(
    class_names,
    per_class_accuracy
)

plt.ylim(0, 1)

plt.ylabel("Accuracy")

plt.title(
    "Per-Class Accuracy"
)

for bar, score in zip(bars, per_class_accuracy):

    plt.text(

        bar.get_x() + bar.get_width()/2,

        score + 0.01,

        f"{score:.3f}",

        ha='center'
    )

plt.tight_layout()

plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "per_class_accuracy.png"
    ),

    dpi=300
)

plt.close()

# ============================================================
# TRAINING CURVES
# ============================================================

history_df = pd.DataFrame(history)

history_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "training_log.csv"
    ),

    index=False
)

# ------------------------------------------------------------
# LOSS CURVE
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Train Loss"
)

plt.plot(
    history_df["epoch"],
    history_df["test_loss"],
    label="Test Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Loss Curve")

plt.legend()

plt.tight_layout()

plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "loss_curve.png"
    ),

    dpi=300
)

plt.close()

# ------------------------------------------------------------
# F1 CURVE
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    history_df["epoch"],
    history_df["weighted_f1"],
    label="Weighted F1"
)

plt.plot(
    history_df["epoch"],
    history_df["macro_f1"],
    label="Macro F1"
)

plt.xlabel("Epoch")
plt.ylabel("F1 Score")

plt.title("F1 Score Curve")

plt.legend()

plt.tight_layout()

plt.savefig(

    os.path.join(
        OUTPUT_DIR,
        "f1_curve.png"
    ),

    dpi=300
)

plt.close()

# ============================================================
# INFERENCE TIME
# ============================================================

dummy_input = torch.randn(
    1,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
).to(DEVICE)

model.eval()

num_runs = 100

starter = time.time()

with torch.no_grad():

    for _ in range(num_runs):

        _ = model(dummy_input)

ender = time.time()

avg_inference_time = (
    (ender - starter) / num_runs
)

# ============================================================
# PARAMETER COUNT
# ============================================================

total_params, trainable_params = count_parameters(
    model
)

# ============================================================
# TOTAL TRAINING TIME
# ============================================================

total_training_time = (
    time.time() - start_training_time
)

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n================================================")
print("FINAL RESULTS")
print("================================================")

print(f"\nFinal Accuracy      : {test_metrics['accuracy']:.4f}")

print(f"\nWeighted F1         : {test_metrics['weighted_f1']:.4f}")

print(f"\nMacro F1            : {test_metrics['macro_f1']:.4f}")

print("\n------------------------------------------------")

print(f"\nTotal Parameters    : {total_params:,}")

print(f"\nTrainable Params    : {trainable_params:,}")

print(
    f"\nInference Time      : "
    f"{avg_inference_time:.6f} sec/image"
)

print(
    f"\nTraining Time       : "
    f"{total_training_time/60:.2f} minutes"
)

print("\n================================================")
print("Experiment Completed Successfully")
print("================================================")

print(f"\nResults saved to: {OUTPUT_DIR}")


Starting Advanced Fine-Tuning

Epoch [1/35]


Train Loss: 1.4491 | Test Accuracy: 0.0155 | Weighted F1: 0.0005

Epoch [2/35]


Train Loss: 1.4185 | Test Accuracy: 0.0155 | Weighted F1: 0.0005

Epoch [3/35]


Train Loss: 1.4182 | Test Accuracy: 0.0155 | Weighted F1: 0.0005

Epoch [4/35]

Partial unfreezing...



Train Loss: 1.3887 | Test Accuracy: 0.0155 | Weighted F1: 0.0005

Epoch [5/35]


...

Train Loss: 1.0992 | Test Accuracy: 0.1864 | Weighted F1: 0.1484

Epoch [33/35]


Train Loss: 1.0921 | Test Accuracy: 0.1943 | Weighted F1: 0.1552

Epoch [34/35]


Train Loss: 1.1042 | Test Accuracy: 0.1996 | Weighted F1: 0.1579

Epoch [35/35]


Train Loss: 1.0647 | Test Accuracy: 0.2172 | Weighted F1: 0.1739

Final Evaluation


              precision    recall  f1-score   support

       angry     0.3023    0.0271    0.0498       958
   disgusted     0.0364    0.9189    0.0700       111
     fearful     0.1791    0.0117    0.0220      1024
       happy     0.0000    0.0000    0.0000      1774
     neutral     0.2795    0.4071    0.3315      1233
         sad     0.2924    0.2045    0.2407      1247
   surprised     0.4260    0.7966    0.5551       831

    accuracy                         0.2172      7178
   macro avg     0.2165    0.3380    0.1813      7178
weighted avg     0.2146    0.2172    0.1739      7178


FINAL RESULTS

Final Accuracy      : 0.2172

Weighted F1         : 0.1739

Macro F1            : 0.1813

------------------------------------------------

Total Parameters    : 4,798,083

Trainable Params    : 4,798,083

Inference Time      : 0.009115 sec/image

Training Time       : 62.32 minutes

Experiment Completed Successfully

Results saved to: ./EfficientNetB0_AdvancedFineTune_outputs
